
## 一、核心业务概念：同期群分析（Cohort Analysis）
### 1. 定义
同期群分析是一种**时间维度**的用户分层方法，将用户按**首次行为时间**（如首次消费、首次注册）划分为不同群体，追踪每个群体在后续时间点的表现（如留存率、客单价）。

### 2. 核心指标
- **新增用户数**：某月份首次产生行为的用户数量。
- **留存率**：某群体在后续第N个周期内仍有行为的用户比例。
  $$
  \text{第N月留存率} = \frac{\text{第N月仍有行为的用户数}}{\text{当月新增用户数}} \times 100\%
  $$

### 3. 分析视角（案例：餐厅/流媒体）
| 视角 | 指标 | 业务含义 |
|------|------|----------|
| **留存视角** | 留存人数/率 | 用户活跃度、忠诚度、产品粘性 |
| **价值视角** | 平均客单价 | 用户贡献度、消费意愿 |

---


## 二、Python实现核心代码
### 1. 环境准备与数据读取

In [36]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rc('font',**{'family':'Microsoft YaHei,SimHei'})
df = pd.read_csv('customers_data.csv')
df['付款时间'] = pd.to_datetime(df['付款时间'])
df['付款年月'] = df['付款时间'].dt.strftime('%Y-%m')
df.sample(5)


,脱敏客户ID,付款时间,支付金额,付款年月
264,cumid260,2023-09-05 01:05:01,103.11,2023-09
36220,cumid28201,2024-02-19 15:19:43,27.50,2024-02
28372,cumid22556,2024-01-19 21:04:20,160.00,2024-01
37021,cumid28804,2024-02-21 16:28:08,115.57,2024-02
32445,cumid25487,2024-02-06 18:51:03,142.49,2024-02


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40181 entries, 0 to 40180
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   脱敏客户ID  40181 non-null  object        
 1   付款时间    40181 non-null  datetime64[ns]
 2   支付金额    40181 non-null  float64       
 3   付款年月    40181 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 1.2+ MB


#### 2.单月新增和留存情况

###  关键函数：np.intersect1d & np.setdiff1d
| 函数 | 作用 | 应用场景 |
|------|------|----------|
| **np.intersect1d(arr1, arr2)** | 求交集，去重 | 统计**留存用户**（本月也在下月消费的用户） |
| **np.setdiff1d(arr1, arr2)** | 求差集，去重 | 统计**新增用户**（下月有但上月无的用户） |

In [31]:
# 2023年9月用户新增情况
Sep_new = df.query('付款年月 == "2023-09"')
print(f'2023-09消费记录数:{len(Sep_new)},新增用户数（唯一ID）:{len(Sep_new["脱敏客户ID"].unique())}')

#9月新增的用户中，有多少留存到了10月
#与历史数据做匹配，即客户昵称在2023年10月且也在2023年9月的
month = '2023-10'
month_customer = df[df['付款年月'] == month]
common_users = np.intersect1d(Sep_new['脱敏客户ID'],month_customer['脱敏客户ID'])
print(f'{month}的客户中，有{len(common_users)} 个是上个月留存下来的')

2023-09消费记录数:2187,新增用户数（唯一ID）:2031
2023-10的客户中，有252 个是上个月留存下来的


In [32]:
# 循环构造
print('2023-09的客户在后续约分钟的留存情况...')

stay = []
for i in ['2023-10','2023-11']:
    next_month = df[df['付款年月'] == i]
    #2023-09 的客户还出现在时间i的DataFrame中
    common_users = np.intersect1d(Sep_new["脱敏客户ID"],next_month["脱敏客户ID"])
    stay.append([i+'留存人数:',len(common_users)])
stay

2023-09的客户在后续约分钟的留存情况...


[['2023-10留存人数:', 252], ['2023-11留存人数:', 216]]

### 3.循环构建每个月的新增和留存

In [33]:
month_list = df['付款年月'].unique()
for i in range(0,len(month_list)-1):
    #len()-1的原因：最后一个月之后就没有数据了
    # 筛选出month_list中的每月消费，并统计客户数量
    print(f'下面统计:{month_list[i]}的新增情况...')
    current_data = df[df['付款年月'] == month_list[i]]
    current_clients = current_data['脱敏客户ID'].unique()

    # ===================统计新增情况=======================
    # 跳过数据集中的第一个月，因为没有历史数据来验证该客户是否为新增客户
    if i == 0 :
        print(f'{month_list[i]}是第一个月，无需验证客户是否为新增客户')
        new_client_num = len(current_clients)
        print(f'该月的新增用户数为：{new_client_num}')
    else:
        #筛选该月(current_month)之前的所有历史消费记录，核心代码
        history_month = month_list[:i]
        print(f'{month_list[i]}的历史年月为;{history_month}')
        history_data = df[df['付款年月'].isin(history_month)]
        #筛选未在历史消费中出现过的新增客户,核心代码
        new_users = np.setdiff1d(current_data['脱敏客户ID'],history_data['脱敏客户ID'])
        print(f'相较于历史年月，该月的新增客户数为:{len(new_users)}')

    print('\n')

下面统计:2023-09的新增情况...
2023-09是第一个月，无需验证客户是否为新增客户
该月的新增用户数为：2031


下面统计:2023-10的新增情况...
2023-10的历史年月为;['2023-09']
相较于历史年月，该月的新增客户数为:7043


下面统计:2023-11的新增情况...
2023-11的历史年月为;['2023-09' '2023-10']
相较于历史年月，该月的新增客户数为:4732


下面统计:2023-12的新增情况...
2023-12的历史年月为;['2023-09' '2023-10' '2023-11']
相较于历史年月，该月的新增客户数为:4979


下面统计:2024-01的新增情况...
2024-01的历史年月为;['2023-09' '2023-10' '2023-11' '2023-12']
相较于历史年月，该月的新增客户数为:5110




In [34]:
for i in range(0,len(month_list)-1):
    # len()-1的原因：最后一个月之后就没有数据了
    # 筛选出month_list中的每月消费，并统计客户数量
    print(f'下面统计:{month_list[i]}的新增情况...')
    current_data = df[df['付款年月']==month_list[i]]
    current_clients = current_data['脱敏客户ID'].unique()
    # =================统计新增情况=========================
    # 跳过数据集中的第一个月，因为没有历史数据来验证该客户是否为新增客户
    if i ==0:
        print(f'{month_list[i]}是第一个月，无需验证客户是否为新增客户')
        new_client_num = len(current_clients)
        print(f'该月的新增用户数为：{new_client_num}')
    else:
        #筛选该月(current_month)之前的所有历史消费记录，核心代码
        history_month = month_list[:i]
        print(f'{month_list[i]}的历史年月为;{history_month}')
        history_data = df[df['付款年月'].isin(history_month)]
        #筛选未在历史消费中出现过的新增客户,核心代码
        new_users = np.setdiff1d(current_data['脱敏客户ID'],history_data['脱敏客户ID'])
        print(f'相较于历史年月，该月的新增客户数为:{len(new_users)}')
    #====================统计留存情况============================
    print('-'*50)
    print('下面统计该月之后的每个月留存情况...')
    for j in range(i+1,len(month_list)):
        next_month_data = df[df['付款年月'] == month_list[j]]
        #统计即出现在该月又出现在下个月的用户
        next_month_retain = np.intersect1d(current_data['脱敏客户ID'],next_month_data['脱敏客户ID'])
        print(f'{month_list[j]}的留存人数:{len(next_month_retain)}')
    print('\n')

下面统计:2023-09的新增情况...
2023-09是第一个月，无需验证客户是否为新增客户
该月的新增用户数为：2031
--------------------------------------------------
下面统计该月之后的每个月留存情况...
2023-10的留存人数:252
2023-11的留存人数:216
2023-12的留存人数:163
2024-01的留存人数:156
2024-02的留存人数:164


下面统计:2023-10的新增情况...
2023-10的历史年月为;['2023-09']
相较于历史年月，该月的新增客户数为:7043
--------------------------------------------------
下面统计该月之后的每个月留存情况...
2023-11的留存人数:623
2023-12的留存人数:491
2024-01的留存人数:488
2024-02的留存人数:491


下面统计:2023-11的新增情况...
2023-11的历史年月为;['2023-09' '2023-10']
相较于历史年月，该月的新增客户数为:4732
--------------------------------------------------
下面统计该月之后的每个月留存情况...
2023-12的留存人数:637
2024-01的留存人数:562
2024-02的留存人数:486


下面统计:2023-12的新增情况...
2023-12的历史年月为;['2023-09' '2023-10' '2023-11']
相较于历史年月，该月的新增客户数为:4979
--------------------------------------------------
下面统计该月之后的每个月留存情况...
2024-01的留存人数:821
2024-02的留存人数:636


下面统计:2024-01的新增情况...
2024-01的历史年月为;['2023-09' '2023-10' '2023-11' '2023-12']
相较于历史年月，该月的新增客户数为:5110
--------------------------------------------------
下面统计该月之后的每个月留存情况

In [20]:
import pandas as pd

# 1. 准备数据
data = {
    "月份": ["2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02"],
    "当月新增": [2031, 7043, 4732, 4979, 5110, 7101],
    "+1月": [252, 623, 637, 821, 909, "-"],
    "+2月": [216, 491, 562, 636, "-", "-"],
    "+3月": [163, 488, 486, "-", "-", "-"],
    "+4月": [156, 491, "-", "-", "-", "-"],
    "+5月": [164, "-", "-", "-", "-", "-"]
}

# 2. 生成 DataFrame
df_retention = pd.DataFrame(data)

df_retention

,月份,当月新增,+1月,+2月,+3月,+4月,+5月
0,2023-09,2031,252,216,163,156,164
1,2023-10,7043,623,491,488,491,-
2,2023-11,4732,637,562,486,-,-
3,2023-12,4979,821,636,-,-,-
4,2024-01,5110,909,-,-,-,-
5,2024-02,7101,-,-,-,-,-


In [21]:
import pandas as pd

# 原始数据
data = {
    "月份": ["2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02"],
    "当月新增": [2031, 7043, 4732, 4979, 5110, 7101],
    "+1月": [252, 623, 637, 821, 909, None],
    "+2月": [216, 491, 562, 636, None, None],
    "+3月": [163, 488, 486, None, None, None],
    "+4月": [156, 491, None, None, None, None],
    "+5月": [164, None, None, None, None, None]
}

df = pd.DataFrame(data)

# 计算留存率：留存人数 / 当月新增，转成百分比并保留整数
for col in ["+1月", "+2月", "+3月", "+4月", "+5月"]:
    df[col] = (df[col] / df["当月新增"] * 100).round(0).astype('Int64').astype(str) + "%"
    df[col] = df[col].replace("nan%", "-")  # 无数据用"-"代替

print(df.to_string(index=False))

     月份  当月新增   +1月   +2月   +3月   +4月   +5月
2023-09  2031   12%   11%    8%    8%    8%
2023-10  7043    9%    7%    7%    7% <NA>%
2023-11  4732   13%   12%   10% <NA>% <NA>%
2023-12  4979   16%   13% <NA>% <NA>% <NA>%
2024-01  5110   18% <NA>% <NA>% <NA>% <NA>%
2024-02  7101 <NA>% <NA>% <NA>% <NA>% <NA>%


In [22]:
df

,月份,当月新增,+1月,+2月,+3月,+4月,+5月
0,2023-09,2031,12%,11%,8%,8%,8%
1,2023-10,7043,9%,7%,7%,7%,<NA>%
2,2023-11,4732,13%,12%,10%,<NA>%,<NA>%
3,2023-12,4979,16%,13%,<NA>%,<NA>%,<NA>%
4,2024-01,5110,18%,<NA>%,<NA>%,<NA>%,<NA>%
5,2024-02,7101,<NA>%,<NA>%,<NA>%,<NA>%,<NA>%



1. **整体留存趋势分析**
    - **2023年9月同期群**：当月新增2031人，次月（+1月）留存252人，+2月留存216人，+3月163人，+4月156人，+5月164人。这个同期群的留存曲线相对平缓，说明这批顾客的长期粘性较好，虽然次月有明显流失，但后续几个月的流失速度减缓，甚至在第5个月有小幅回升（可能是促销或节日效应）。
    - **2023年10月同期群**：当月新增7043人，是所有同期群中新增人数最多的。次月留存623人，+2月491人，+3月488人，+4月491人。这个同期群的留存人数绝对值较高，但留存率（留存人数/新增人数）较低，次月留存率约为8.8%，后续稳定在7%左右。这表明虽然新增规模大，但顾客粘性一般。
    - **2023年11月同期群**：当月新增4732人，次月留存637人（留存率约13.5%），+2月562人，+3月486人。这个同期群的留存率相对较高，尤其是次月留存表现较好，说明11月吸引的顾客质量较高，或者当月有较强的促活措施。
    - **2023年12月同期群**：当月新增4979人，次月留存821人（留存率约16.5%），+2月636人。12月是节假日密集的月份，新增顾客的留存率明显高于其他月份，说明节日效应显著提升了顾客的复购意愿。
    - **2024年1月同期群**：当月新增5110人，次月留存909人（留存率约17.8%），是所有同期群中次月留存率最高的。这可能与春节前的消费高峰有关，顾客在节前消费后，节后仍有较高的复购意愿。
    - **2024年2月同期群**：当月新增7101人，是所有同期群中新增人数最高的，但次月数据缺失（显示为“-”），可能是数据尚未更新或该同期群的后续消费数据还未统计。从新增人数来看，2月可能是餐厅的旺季，但需要关注后续的留存表现。

2. **留存率变化趋势**
    - **次月留存率**：从2023年9月到2024年1月，次月留存率呈现上升趋势，从12.4%（252/2031）上升到17.8%（909/5110）。这说明餐厅的顾客留存策略在逐步优化，或者顾客对餐厅的满意度在提升。
    - **长期留存率**：从+2月到+5月，各同期群的留存人数逐渐减少，但减少的速度在放缓。例如，2023年9月同期群在+4月和+5月的留存人数基本持平，说明这批顾客已经形成了稳定的消费习惯。

3. **业务建议**
    - **优化新客转化**：虽然2023年10月和2024年2月的新增顾客数量较多，但留存率相对较低。建议针对这两个月的新增顾客进行深入分析，找出流失原因，比如是否是因为促销活动结束后顾客不再消费，或者是因为服务质量问题。
    - **加强节日营销**：2023年12月和2024年1月的留存率较高，说明节日效应对顾客留存有显著影响。建议在未来的节假日期间加大营销力度，比如推出节日专属套餐、会员优惠等，以提升顾客的复购率。
    - **提升顾客粘性**：对于留存率较高的同期群（如2023年11月和2024年1月），可以分析这些顾客的消费特征，比如消费时间、消费金额、菜品偏好等，针对性地推出个性化推荐和会员服务，进一步提升顾客的忠诚度。